In [1]:
import pandas as pd
import torch
import torch.nn as nn
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler

In [2]:
train = pd.read_csv(r"dataset/train.csv")
test = pd.read_csv(r"dataset/test.csv")

In [3]:
len(train),len(test)

(750000, 250000)

In [4]:
train.head(3)

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0


In [5]:
test.head(3)

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,750000,male,45,177.0,81.0,7.0,87.0,39.8
1,750001,male,26,200.0,97.0,20.0,101.0,40.5
2,750002,female,29,188.0,85.0,16.0,102.0,40.4


In [6]:
train.isnull().sum()

id            0
Sex           0
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
Calories      0
dtype: int64

In [7]:
ohe = OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")
encoded = ohe.fit_transform(train[["Sex"]])

In [8]:
encoded_column = ohe.get_feature_names_out()
encoded_column

array(['Sex_male'], dtype=object)

In [9]:
encoded_df = pd.DataFrame(encoded,columns=encoded_column)
encoded_df.head(2)

,Sex_male
0,1.0
1,0.0


In [10]:
joblib.dump(ohe,filename="OneHotEncoder.pkl")

['OneHotEncoder.pkl']

In [11]:
final_df = pd.concat(
    [train.drop("Sex",axis=1),encoded_df],axis=1
)

In [12]:
final_df

,id,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Sex_male
0,0,36,189.0,82.0,26.0,101.0,41.0,150.0,1.0
1,1,64,163.0,60.0,8.0,85.0,39.7,34.0,0.0
2,2,51,161.0,64.0,7.0,84.0,39.8,29.0,0.0
3,3,20,192.0,90.0,25.0,105.0,40.7,140.0,1.0
4,4,38,166.0,61.0,25.0,102.0,40.6,146.0,0.0
...,...,...,...,...,...,...,...,...,...
749995,749995,28,193.0,97.0,30.0,114.0,40.9,230.0,1.0
749996,749996,64,165.0,63.0,18.0,92.0,40.5,96.0,0.0
749997,749997,60,162.0,67.0,29.0,113.0,40.9,221.0,1.0
749998,749998,45,182.0,91.0,17.0,102.0,40.3,109.0,1.0


In [13]:
x = final_df.drop("Calories",axis=1).values
y = final_df["Calories"].values

In [14]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [15]:
x_train = torch.tensor(x_train,dtype=torch.float32)
x_test = torch.tensor(x_test,dtype=torch.float32)
y_train = torch.tensor(y_train,dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test,dtype=torch.float32).view(-1,1)

In [16]:
scaler_1 = StandardScaler()
x_train = scaler_1.fit_transform(x_train)
x_test = scaler_1.transform(x_test)

In [17]:
scaler_2 = StandardScaler()
y_train = scaler_2.fit_transform(y_train)
y_test = scaler_2.transform(y_test)

In [18]:
joblib.dump(scaler_1,filename="x_train_x_test.pkl")
joblib.dump(scaler_2,filename="y_train_y_test.pkl")

['y_train_y_test.pkl']

In [19]:
model = nn.Sequential(
    nn.Linear(in_features=8,out_features=128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.3),
    
    nn.Linear(in_features=128,out_features=64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(in_features=64,out_features=32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(32,1)
)

In [20]:
model.training

True

In [21]:
loss_function = nn.MSELoss()

In [22]:
optimizer_function = torch.optim.Adam(model.parameters(),lr=0.001)

In [23]:
x_train = torch.tensor(x_train,dtype=torch.float32)

In [24]:
y_train = torch.tensor(y_train,dtype=torch.float32)

In [25]:
epochs = 200
progress_bar = tqdm(range(epochs))
for epoch in progress_bar:
    model.train()
    optimizer_function.zero_grad()
    predict = model(x_train)
    loss = loss_function(predict,y_train)
    loss.backward()
    optimizer_function.step()
    

    torch.save({
        "epochs":epoch,
        "model":model.state_dict(),
        "optimizer_function":optimizer_function.state_dict(),
        "loss":loss.item()
    },f"Checkpoints_{epoch}.pth")

    progress_bar.set_postfix({
        "Epoch":epoch,
        "Loss":f"Loss {loss.item():.4f}"
    })


100%|██████████| 200/200 [13:12<00:00,  3.96s/it, Epoch=199, Loss=Loss 0.0556]
